# MAJ-Debate: Stage 3 Argumentation Framework Engine (Fast)

Computes grounded and preferred extensions of Dung argumentation frameworks from Stage 2 output.

**Speed rewrite highlights:**

- Grounded extension now O(|args| + |edges|) via iterative labeling, no repeated scans.
- Preferred extensions use a proper enumeration (modified Nielsen-Parsons branch-and-bound) that prunes the search tree — O(2^N) worst case but typically a few hundred nodes even for N=30.
- Per-topic `attackers[x]` and `attacks_by[x]` dicts are built once; all helpers use those instead of scanning edge lists.
- SCC decomposition: preferred extensions are computed per strongly-connected-component then combined, which breaks most topics into tiny sub-problems.
- Hard time budget per topic to prevent pathological graphs from stalling the run.

## 0. Imports & Configuration

In [1]:
import os, json, statistics, time
from pathlib import Path
from datetime import datetime
from collections import defaultdict

try:
    import networkx as nx
    _HAS_NX = True
except ImportError:
    _HAS_NX = False
    print('NetworkX not installed - SCC decomposition disabled. pip install networkx for speedup.')

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://YOUR_VM_IP:5000')
TOPIC_LIMIT = int(os.environ.get('MAJ_STAGE3_TOPIC_LIMIT', '0'))
INCLUDE_SUPPORT = os.environ.get('MAJ_STAGE3_INCLUDE_SUPPORT', '1') == '1'
COMPUTE_PREFERRED_ONLY_IF_NEEDED = os.environ.get('MAJ_STAGE3_PREFERRED_ONLY_IF_NEEDED', '1') == '1'
EVAL_SPLIT = os.environ.get('MAJ_EVAL_SPLIT', 'ddo_sample')
PREFERRED_TIME_BUDGET_SEC = float(os.environ.get('MAJ_STAGE3_PREFERRED_TIME_BUDGET', '15'))
PREFERRED_MAX_ARGS = int(os.environ.get('MAJ_STAGE3_PREFERRED_MAX_ARGS', '30'))

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists() or (p / 'outputs').exists()), cwd)
IN_FILE = PROJECT_ROOT / 'outputs' / 'stage2' / EVAL_SPLIT / 'stage2_relations.json'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'stage3' / EVAL_SPLIT
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / 'stage3_graphs.json'
FAILURES_FILE = OUT_DIR / 'stage3_failures.json'
RESUME_EXISTING = os.environ.get('MAJ_STAGE3_RESUME', '1').strip() not in {'0', 'false', 'False'}
CONTINUE_ON_ERROR = os.environ.get('MAJ_STAGE3_CONTINUE_ON_ERROR', '1').strip() not in {'0', 'false', 'False'}

print(f'Input              : {IN_FILE.resolve()}')
print(f'Output             : {OUT_FILE.resolve()}')
print(f'Failures           : {FAILURES_FILE.resolve()}')
print(f'Resume             : {RESUME_EXISTING}')
print(f'Continue on error  : {CONTINUE_ON_ERROR}')
print(f'Preferred only if needed : {COMPUTE_PREFERRED_ONLY_IF_NEEDED}')
print(f'Preferred time budget    : {PREFERRED_TIME_BUDGET_SEC}s per topic')
print(f'Preferred max args       : {PREFERRED_MAX_ARGS} (topics over this fall back to grounded)')
print(f'NetworkX available       : {_HAS_NX}')

Input              : /home/jupyter-st125974-ml/Project/outputs/stage2/ddo_sample/stage2_relations.json
Output             : /home/jupyter-st125974-ml/Project/outputs/stage3/ddo_sample/stage3_graphs.json
Failures           : /home/jupyter-st125974-ml/Project/outputs/stage3/ddo_sample/stage3_failures.json
Resume             : True
Continue on error  : True
Preferred only if needed : True
Preferred time budget    : 15.0s per topic
Preferred max args       : 30 (topics over this fall back to grounded)
NetworkX available       : True


## 1. Dung Semantics — Fast Implementations

Key types:

- `args`: a list/set of argument IDs
- `attackers`: dict `{x: set_of_args_attacking_x}`
- `attacks_by`: dict `{x: set_of_args_attacked_by_x}`
- Admissible set: conflict-free + every member is defended by the set
- Grounded: least fixed-point of the characteristic function (deterministic, unique)
- Preferred: maximal admissible sets (may be multiple)

In [2]:
def build_adjacency(args, attack_edges):
    """Build attacker/attacked-by dicts from edges. O(|edges|)."""
    attackers = {a: set() for a in args}
    attacks_by = {a: set() for a in args}
    for src, tgt in attack_edges:
        if src in attackers and tgt in attackers:
            attackers[tgt].add(src)
            attacks_by[src].add(tgt)
    return attackers, attacks_by


def grounded_extension_fast(args, attackers, attacks_by):
    """Grounded extension via iterative labeling. O(|args| + |edges|).

    A node is IN if all its attackers are OUT.
    A node is OUT if some attacker is IN.
    Unlabeled nodes are UNDEC. Returns the IN set.
    """
    labels = {a: 'UNDEC' for a in args}
    remaining_attackers = {a: len(attackers[a]) for a in args}
    # Nodes with no attackers are definitely IN
    ready = [a for a in args if remaining_attackers[a] == 0]
    while ready:
        next_ready = []
        # Mark these as IN, propagate OUT to their targets
        for a in ready:
            if labels[a] == 'UNDEC':
                labels[a] = 'IN'
                for tgt in attacks_by[a]:
                    if labels[tgt] == 'UNDEC':
                        labels[tgt] = 'OUT'
                        # When a node goes OUT, decrement remaining_attackers
                        # of everything it attacks
                        for tgt2 in attacks_by[tgt]:
                            if labels[tgt2] == 'UNDEC':
                                remaining_attackers[tgt2] -= 1
                                if remaining_attackers[tgt2] == 0:
                                    next_ready.append(tgt2)
        ready = next_ready
    return {a for a, lbl in labels.items() if lbl == 'IN'}


def is_conflict_free(candidate_set, attackers):
    """Check conflict-freeness. O(|candidate_set|)."""
    for a in candidate_set:
        if attackers[a] & candidate_set:
            return False
    return True


def defends(candidate_set, target, attackers, attacks_by):
    """True iff candidate_set defends target (attacks every attacker of target)."""
    for atk in attackers[target]:
        if not (attackers[atk] & candidate_set):
            return False
    return True


def is_admissible(candidate_set, attackers, attacks_by):
    if not is_conflict_free(candidate_set, attackers):
        return False
    for a in candidate_set:
        if not defends(candidate_set, a, attackers, attacks_by):
            return False
    return True

## 2. Preferred Extensions — Branch-and-Bound with SCC Decomposition

Two-level speedup over the naive powerset approach:

1. **SCC decomposition:** an argumentation framework's attack graph can be split into strongly-connected components. Preferred extensions of the whole graph are the combinations of preferred extensions of each SCC (treating inter-SCC attacks as context). Most real debates have many small SCCs.
2. **Branch-and-bound within each SCC:** enumerate maximal admissible sets directly. At each step, pick an unlabeled argument; either include it (and mark its attackers as excluded) or exclude it. Prune branches that can't possibly yield maximal admissible sets.

In [3]:
def preferred_extensions_bb(args, attackers, attacks_by, deadline=None):
    """Enumerate all preferred (maximal admissible) extensions.

    Uses branch-and-bound. Aborts if deadline exceeded and returns partial results.

    Returns list of sorted lists (sorted for determinism).
    """
    args = list(args)
    args_set = set(args)

    # Trivial case: empty graph
    if not args:
        return [[]], False

    # Sort arguments by degree (highest in-degree first) - this tends to
    # branch on constrained variables first, pruning the tree faster.
    args.sort(key=lambda a: -(len(attackers[a]) + len(attacks_by[a])))

    results = []
    timeout_flag = [False]

    def branch(idx, included, excluded):
        if deadline is not None and time.monotonic() > deadline:
            timeout_flag[0] = True
            return
        if idx == len(args):
            if is_admissible(included, attackers, attacks_by):
                results.append(frozenset(included))
            return
        a = args[idx]
        if a in included or a in excluded:
            branch(idx + 1, included, excluded)
            return
        # Branch 1: include a (only if conflict-free with existing included)
        if not (attackers[a] & included) and not (attacks_by[a] & included):
            new_excluded = excluded | attackers[a]
            new_included = included | {a}
            branch(idx + 1, new_included, new_excluded)
        # Branch 2: exclude a
        branch(idx + 1, included, excluded | {a})

    branch(0, frozenset(), frozenset())

    # Filter to maximal only. Dedup by frozenset equality.
    unique = set(results)
    maximal = []
    unique_list = list(unique)
    for i, s in enumerate(unique_list):
        is_max = True
        for j, other in enumerate(unique_list):
            if i != j and s < other:
                is_max = False
                break
        if is_max:
            maximal.append(sorted(s))

    # Sort for determinism
    maximal.sort()
    return maximal, timeout_flag[0]


def preferred_extensions_by_scc(args, attackers, attacks_by, deadline=None):
    """Compute preferred extensions by decomposing into SCCs.

    If NetworkX available, decomposes the attack graph into strongly-connected
    components, computes preferred extensions within each component given the
    context of already-decided components, and combines.
    """
    args_set = set(args)
    if not _HAS_NX or len(args_set) <= 8:
        # Small enough to brute force without decomposition
        return preferred_extensions_bb(args_set, attackers, attacks_by, deadline=deadline)

    G = nx.DiGraph()
    G.add_nodes_from(args_set)
    for src, targets in attacks_by.items():
        for t in targets:
            G.add_edge(src, t)

    # Condensation: each SCC becomes one node. Process in topological order.
    sccs = list(nx.strongly_connected_components(G))
    if len(sccs) == 1:
        # Can't decompose further
        return preferred_extensions_bb(args_set, attackers, attacks_by, deadline=deadline)

    condensation = nx.condensation(G, scc=sccs)
    scc_order = list(nx.topological_sort(condensation))
    scc_nodes = {i: sccs[i] for i in range(len(sccs))}

    # Accumulator: list of sets of (accepted, rejected) arg-ids from already-processed SCCs
    partial_extensions = [(frozenset(), frozenset())]
    timeout = False

    for scc_idx in scc_order:
        if deadline is not None and time.monotonic() > deadline:
            timeout = True
            break
        scc = scc_nodes[scc_idx]
        new_partials = []
        for accepted, rejected in partial_extensions:
            # Within this SCC: attackers from already-accepted nodes make
            # their targets OUT (rejected). Attackers from rejected nodes
            # don't constrain.
            forced_out = set()
            for a in scc:
                if attackers[a] & accepted:
                    forced_out.add(a)
            # Nodes in SCC attacked by accepted are pre-rejected.
            # Compute preferred extensions of this SCC subgraph,
            # treating forced_out as excluded.
            sub_args = scc - forced_out
            if not sub_args:
                # Nothing to decide here
                new_partials.append((accepted, rejected | forced_out))
                continue
            sub_attackers = {a: attackers[a] & sub_args for a in sub_args}
            sub_attacks_by = {a: attacks_by[a] & sub_args for a in sub_args}
            sub_exts, to = preferred_extensions_bb(sub_args, sub_attackers, sub_attacks_by, deadline=deadline)
            if to:
                timeout = True
            if not sub_exts:
                new_partials.append((accepted, rejected | forced_out))
                continue
            for sub_ext in sub_exts:
                sub_ext_set = frozenset(sub_ext)
                scc_rejected = sub_args - sub_ext_set
                new_partials.append((
                    accepted | sub_ext_set,
                    rejected | forced_out | scc_rejected,
                ))
        partial_extensions = new_partials
        # Prune duplicates after each SCC (rare but possible)
        seen = set()
        dedup = []
        for acc, rej in partial_extensions:
            key = (acc, rej)
            if key not in seen:
                seen.add(key)
                dedup.append((acc, rej))
        partial_extensions = dedup

    # Extract just the accepted sets, validate each as truly admissible
    result = []
    for accepted, _ in partial_extensions:
        if is_admissible(accepted, attackers, attacks_by):
            result.append(sorted(accepted))
    # Dedup + sort for determinism
    seen_keys = set()
    unique = []
    for ext in result:
        key = tuple(ext)
        if key not in seen_keys:
            seen_keys.add(key)
            unique.append(ext)
    unique.sort()
    return unique, timeout


def side_vote(extension, arg_index):
    counts = {'PRO': 0, 'CON': 0}
    for arg_id in extension:
        stance = arg_index[arg_id].get('stance')
        if stance in counts:
            counts[stance] += 1
    if counts['PRO'] > counts['CON']:
        return 'PRO'
    if counts['CON'] > counts['PRO']:
        return 'CON'
    return 'UNDECIDED'

## 3. Topic-Level Graph Construction

Builds the argumentation framework for one topic, runs both semantics, and packages the output.

In [4]:
def load_stage2(path=IN_FILE):
    if not path.exists():
        raise FileNotFoundError(f'Stage 2 input not found: {path}')
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def relation_justification(rel):
    text = rel.get('justification')
    if isinstance(text, str) and text.strip():
        return text.strip()
    label = rel.get('label', 'relation')
    conf = rel.get('confidence')
    premise = rel.get('attacked_premise')
    if premise:
        return f'{label} relation inferred from Stage 2 output; attacked premise: {premise}'
    if isinstance(conf, (int, float)):
        return f'{label} relation inferred from Stage 2 output with confidence {conf:.3f}.'
    return f'{label} relation inferred from Stage 2 output.'


def build_topic_graph(topic_rec, include_support=True):
    t0 = time.monotonic()
    args = topic_rec['arguments']
    arg_index = {a['arg_id']: a for a in args}
    arg_ids = set(arg_index.keys())

    attack_edges = []
    support_edges = []
    for rel in topic_rec['relations']:
        if not rel.get('kept'):
            continue
        edge = {
            'source_arg_id': rel['source_arg_id'],
            'target_arg_id': rel['target_arg_id'],
            'confidence': rel.get('confidence', 0.0),
            'justification': relation_justification(rel),
            'attacked_premise': rel.get('attacked_premise'),
        }
        if rel['label'] == 'Attack':
            attack_edges.append(edge)
        elif include_support and rel['label'] == 'Support':
            support_edges.append(edge)

    # Build once, reuse everywhere
    att_edge_tuples = [(e['source_arg_id'], e['target_arg_id']) for e in attack_edges]
    attackers, attacks_by = build_adjacency(arg_ids, att_edge_tuples)

    # Grounded extension - always fast
    grounded_set = grounded_extension_fast(arg_ids, attackers, attacks_by)
    grounded = sorted(grounded_set)
    grounded_vote = side_vote(grounded, arg_index)

    # Preferred extensions - skip if grounded is already decisive or graph too big
    preferred = []
    fallback_reason = None
    preferred_vote = grounded_vote
    preferred_timeout = False

    need_preferred = (grounded_vote == 'UNDECIDED') or (not COMPUTE_PREFERRED_ONLY_IF_NEEDED)

    if need_preferred:
        if len(arg_ids) > PREFERRED_MAX_ARGS:
            fallback_reason = f'too_many_args_for_preferred_{len(arg_ids)}'
            preferred_vote = grounded_vote
        else:
            if grounded_vote == 'UNDECIDED':
                fallback_reason = 'grounded_winner_unresolved'
            deadline = time.monotonic() + PREFERRED_TIME_BUDGET_SEC
            preferred, preferred_timeout = preferred_extensions_by_scc(
                arg_ids, attackers, attacks_by, deadline=deadline
            )
            if preferred_timeout:
                fallback_reason = (fallback_reason or 'preferred_timeout')
                if not fallback_reason.startswith('preferred_timeout'):
                    fallback_reason += '+preferred_timeout'
            pref_votes = [side_vote(ext, arg_index) for ext in preferred]
            if pref_votes.count('PRO') > pref_votes.count('CON'):
                preferred_vote = 'PRO'
            elif pref_votes.count('CON') > pref_votes.count('PRO'):
                preferred_vote = 'CON'
            else:
                preferred_vote = 'UNDECIDED' if pref_votes else grounded_vote

    chosen_extension = grounded if grounded_vote != 'UNDECIDED' else (preferred[0] if preferred else [])
    chosen_set = set(chosen_extension)
    decisive_edges = [
        e for e in attack_edges
        if e['source_arg_id'] in chosen_set and e['target_arg_id'] not in chosen_set
    ]
    winner = grounded_vote if grounded_vote != 'UNDECIDED' else preferred_vote

    elapsed = time.monotonic() - t0

    return {
        'topic_id': topic_rec['topic_id'],
        'topic_text': topic_rec['topic_text'],
        'domain': topic_rec.get('domain'),
        'benchmark_label': topic_rec.get('benchmark_label'),
        'source_dataset': topic_rec.get('source_dataset'),
        'source_ref': topic_rec.get('source_ref'),
        'evaluation_split': topic_rec.get('evaluation_split', EVAL_SPLIT),
        'run_name': topic_rec.get('run_name'),
        'arguments': args,
        'argument_strength': topic_rec.get('argument_strength', {}),
        'attack_graph': attack_edges,
        'support_graph': support_edges,
        'grounded_extension': grounded,
        'preferred_extensions': preferred,
        'winner': winner,
        'fallback_reason': fallback_reason,
        'decisive_attacks': decisive_edges,
        'summary': {
            'n_arguments': len(args),
            'n_attack_edges': len(attack_edges),
            'n_support_edges': len(support_edges),
            'grounded_extension_size': len(grounded),
            'n_preferred_extensions': len(preferred),
            'winner': winner,
            'graph_stability': round(len(grounded) / max(len(args), 1), 3),
            'disconnected_or_sparse': len(attack_edges) < max(1, len(args) // 3),
            'preferred_timeout': preferred_timeout,
            'elapsed_sec': round(elapsed, 3),
        },
    }


# Optional MLflow
try:
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment('MAJ-Debate')
    MLFLOW_OK = True
except Exception as ex:
    MLFLOW_OK = False
    print(f'MLflow unavailable ({ex}) - results saved locally only')

def mlflow_log(run_name, params, metrics, artifact_paths):
    if not MLFLOW_OK:
        return
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        for path in artifact_paths:
            mlflow.log_artifact(str(path), artifact_path=f'stage3/{EVAL_SPLIT}')

MLflow unavailable (No module named 'mlflow') - results saved locally only


## 4. Main Run Loop

Checkpoints after each topic so you can interrupt and resume.

In [5]:
stage2_doc = load_stage2()
topics = stage2_doc['topics'][:TOPIC_LIMIT] if TOPIC_LIMIT > 0 else stage2_doc['topics']
run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f'stage3-{EVAL_SPLIT}-{run_ts}'

def compute_stage3_summary(topic_payloads):
    if not topic_payloads:
        return {'total_topics': 0, 'total_attack_edges': 0, 'total_support_edges': 0,
                'avg_grounded_extension_size': 0.0, 'avg_graph_stability': 0.0,
                'fallback_count': 0, 'timeout_count': 0, 'avg_elapsed_sec': 0.0}
    return {
        'total_topics': len(topic_payloads),
        'total_attack_edges': sum(t['summary']['n_attack_edges'] for t in topic_payloads),
        'total_support_edges': sum(t['summary']['n_support_edges'] for t in topic_payloads),
        'avg_grounded_extension_size': round(statistics.mean(t['summary']['grounded_extension_size'] for t in topic_payloads), 3),
        'avg_graph_stability': round(statistics.mean(t['summary']['graph_stability'] for t in topic_payloads), 3),
        'fallback_count': sum(1 for t in topic_payloads if t.get('fallback_reason')),
        'timeout_count': sum(1 for t in topic_payloads if t['summary'].get('preferred_timeout')),
        'avg_elapsed_sec': round(statistics.mean(t['summary'].get('elapsed_sec', 0) for t in topic_payloads), 3),
    }


def save_stage3_state(topic_results_by_id, failures):
    ordered_topics = [topic_results_by_id[t['topic_id']] for t in topics if t['topic_id'] in topic_results_by_id]
    output_doc = {
        'stage': 3,
        'run_name': run_name,
        'timestamp': run_ts,
        'config': {
            'evaluation_split': EVAL_SPLIT,
            'include_support': INCLUDE_SUPPORT,
            'preferred_only_if_needed': COMPUTE_PREFERRED_ONLY_IF_NEEDED,
            'preferred_time_budget_sec': PREFERRED_TIME_BUDGET_SEC,
            'preferred_max_args': PREFERRED_MAX_ARGS,
            'source_stage2_run': stage2_doc.get('run_name'),
            'resume_existing': RESUME_EXISTING,
        },
        'topics': ordered_topics,
        'summary': compute_stage3_summary(ordered_topics),
    }
    # Atomic save
    tmp = OUT_FILE.with_suffix('.json.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(output_doc, f, indent=2)
    tmp.replace(OUT_FILE)
    with open(FAILURES_FILE, 'w', encoding='utf-8') as f:
        json.dump({'run_name': run_name, 'timestamp': run_ts, 'failed_topics': failures}, f, indent=2)
    return output_doc


# Resume
existing_topics_by_id = {}
if RESUME_EXISTING and OUT_FILE.exists():
    try:
        with open(OUT_FILE, 'r', encoding='utf-8') as f:
            existing_doc = json.load(f)
        for topic_payload in existing_doc.get('topics', []):
            tid = topic_payload.get('topic_id')
            if tid and topic_payload.get('summary'):
                existing_topics_by_id[tid] = topic_payload
        print(f'Resuming from existing output: {len(existing_topics_by_id)} completed topics found')
    except Exception as ex:
        print(f'Could not resume from existing output ({ex}); starting fresh')
        existing_topics_by_id = {}

existing_failures_by_id = {}
if FAILURES_FILE.exists():
    try:
        with open(FAILURES_FILE, 'r', encoding='utf-8') as f:
            existing_failures_doc = json.load(f)
        for failure_payload in existing_failures_doc.get('failed_topics', []):
            failed_tid = failure_payload.get('topic_id')
            if failed_tid and failed_tid not in existing_topics_by_id:
                existing_failures_by_id[failed_tid] = failure_payload
    except Exception as ex:
        print(f'Could not load existing failures ({ex}); continuing with fresh failure log')
        existing_failures_by_id = {}

topic_results_by_id = dict(existing_topics_by_id)
failures_by_id = dict(existing_failures_by_id)

run_start = time.monotonic()
completed_this_run = 0

# Checkpoint every K topics, not every topic, to reduce I/O overhead
CHECKPOINT_EVERY = int(os.environ.get('MAJ_STAGE3_CHECKPOINT_EVERY', '10'))

for idx, topic in enumerate(topics, start=1):
    topic_id = topic['topic_id']
    if topic_id in topic_results_by_id:
        continue  # resume
    try:
        result = build_topic_graph(topic, include_support=INCLUDE_SUPPORT)
        topic_results_by_id[topic_id] = result
        failures_by_id.pop(topic_id, None)
        completed_this_run += 1
        s = result['summary']
        print(f'[{idx}/{len(topics)}] {topic_id}: '
              f'{s["n_arguments"]}a/{s["n_attack_edges"]}e '
              f'grounded={s["grounded_extension_size"]} '
              f'winner={s["winner"]} '
              f'({s["elapsed_sec"]:.2f}s)'
              + (f' fallback={result["fallback_reason"]}' if result.get('fallback_reason') else ''))
        if completed_this_run % CHECKPOINT_EVERY == 0:
            save_stage3_state(topic_results_by_id, list(failures_by_id.values()))
    except Exception as ex:
        failures_by_id[topic_id] = {
            'topic_id': topic_id,
            'topic_text': topic['topic_text'],
            'error': f'{type(ex).__name__}: {ex}',
            'run_name': run_name,
        }
        print(f'[{idx}/{len(topics)}] {topic_id}: FAILED {type(ex).__name__}: {ex}')
        save_stage3_state(topic_results_by_id, list(failures_by_id.values()))
        if not CONTINUE_ON_ERROR:
            raise

# Final save
output_doc = save_stage3_state(topic_results_by_id, list(failures_by_id.values()))

total_elapsed = time.monotonic() - run_start
print(f'\nRun complete in {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')
print(f'Completed this run: {completed_this_run}')
print(f'Total completed:    {len(topic_results_by_id)}')
print(f'Failed:             {len(failures_by_id)}')
print(f'Summary: {output_doc["summary"]}')
print(f'Saved:   {OUT_FILE.resolve()}')

mlflow_log(
    run_name=run_name,
    params=output_doc['config'],
    metrics={k: float(v) for k, v in output_doc['summary'].items() if isinstance(v, (int, float))},
    artifact_paths=[OUT_FILE, FAILURES_FILE],
)

[1/500] DDO_00423: 30a/307e grounded=0 winner=PRO (0.01s) fallback=grounded_winner_unresolved
[2/500] DDO_00430: 30a/226e grounded=12 winner=PRO (0.00s)
[3/500] DDO_00431: 30a/168e grounded=3 winner=PRO (0.00s)
[4/500] DDO_00437: 30a/238e grounded=0 winner=PRO (0.05s) fallback=grounded_winner_unresolved
[5/500] DDO_00456: 30a/175e grounded=1 winner=CON (0.00s)
[6/500] DDO_00460: 30a/160e grounded=11 winner=PRO (0.00s)
[7/500] DDO_00484: 30a/136e grounded=0 winner=PRO (0.26s) fallback=grounded_winner_unresolved
[8/500] DDO_00519: 30a/198e grounded=10 winner=PRO (0.00s)
[9/500] DDO_00522: 30a/287e grounded=0 winner=PRO (0.06s) fallback=grounded_winner_unresolved
[10/500] DDO_00579: 30a/200e grounded=6 winner=CON (0.01s) fallback=grounded_winner_unresolved
[11/500] DDO_00592: 30a/250e grounded=0 winner=PRO (0.04s) fallback=grounded_winner_unresolved
[12/500] DDO_00600: 30a/299e grounded=0 winner=CON (0.01s) fallback=grounded_winner_unresolved
[13/500] DDO_00615: 30a/220e grounded=0 winner

## 5. Inspect Output

In [6]:
if output_doc['topics']:
    sample = output_doc['topics'][0]
    print('Topic      :', sample['topic_text'][:80])
    print('Winner     :', sample['winner'])
    print('Gold label :', sample.get('benchmark_label'))
    print('Fallback   :', sample.get('fallback_reason'))
    print('Summary    :', sample['summary'])
    print()
    # Agreement with benchmark
    agree = 0
    total = 0
    for t in output_doc['topics']:
        gold = t.get('benchmark_label')
        pred = t['winner']
        if gold and pred != 'UNDECIDED':
            total += 1
            if str(gold).upper() == pred:
                agree += 1
    if total:
        print(f'Decisive agreement with benchmark: {agree}/{total} = {agree/total*100:.1f}%')
    # Distribution of timings
    times = [t['summary'].get('elapsed_sec', 0) for t in output_doc['topics']]
    if times:
        print(f'Timing per topic: min={min(times):.3f}s p50={statistics.median(times):.3f}s max={max(times):.3f}s')

Topic      : A Ban is an Act of War.
Winner     : PRO
Gold label : CON
Fallback   : grounded_winner_unresolved
Summary    : {'n_arguments': 30, 'n_attack_edges': 307, 'n_support_edges': 262, 'grounded_extension_size': 0, 'n_preferred_extensions': 1, 'winner': 'PRO', 'graph_stability': 0.0, 'disconnected_or_sparse': False, 'preferred_timeout': False, 'elapsed_sec': 0.008}

Decisive agreement with benchmark: 237/489 = 48.5%
Timing per topic: min=0.001s p50=0.003s max=0.561s
